# Capacità ed early stopping



## Preparazione in Colab

1. Salva una copia del notebook nel tuo Drive.
2. Esegui la cella seguente e attendi «Preparazione completata».
3. Prosegui dall’alto verso il basso e completa gli esercizi.

La cella carica i dataset e i controlli dal repository del corso. Non devi installare Python né caricare manualmente i CSV. Se la sessione viene riavviata, esegui nuovamente la preparazione.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
from urllib.request import urlopen
from IPython.display import display, Markdown

course_source = 'https://raw.githubusercontent.com/prof-davidaffo/kaggle-learntools/master'
course_root = (Path("/content") if Path("/content").is_dir() else Path.home()) / "ai-lab-colab"
course_root.mkdir(parents=True, exist_ok=True)
bootstrap_path = course_root / "_bootstrap.py"
bootstrap_hash = '24c516651bf205b1c9511923ad4971fdca39a6248568d2b3aa46c198e64d1821'
if not bootstrap_path.is_file() or hashlib.sha256(bootstrap_path.read_bytes()).hexdigest() != bootstrap_hash:
    with urlopen(course_source + "/ai_lab/tools/colab_runtime.py", timeout=90) as response:
        bootstrap_data = response.read()
    if hashlib.sha256(bootstrap_data).hexdigest() != bootstrap_hash:
        raise RuntimeError("Course setup changed. Reopen the latest notebook from the course site.")
    bootstrap_path.write_bytes(bootstrap_data)
bootstrap_spec = importlib.util.spec_from_file_location("ai_lab_colab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(bootstrap_spec)
bootstrap_spec.loader.exec_module(bootstrap)
bootstrap.download_runtime(course_source + "/ai_lab/colab/runtime.json", course_source, course_root)
bootstrap.setup(course_root, globals(), deep_learning=True)
display(Markdown("**Preparazione completata.** Puoi eseguire le celle della lezione."))


# Introduzione
Migliorerai il training con una callback di **early stopping** per limitare l'overfitting. Esegui la cella iniziale.


In [ ]:
# Configura i grafici.
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
# Imposta i valori predefiniti di Matplotlib.
plt.rc('figure', autolayout=True)
plt.rc('axes', labelweight='bold', labelsize='large',
       titleweight='bold', titlesize=18, titlepad=10)
plt.rc('animation', html='html5')

# Configura il sistema di verifica degli esercizi.
from learntools.core import binder
binder.bind(globals())
from learntools.deep_learning_intro.ex4 import *

Carica il dataset **Spotify**. Il target è la popolarità di un brano, da prevedere usando feature audio come **`tempo`**, **`danceability`** e **`mode`**.


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.model_selection import GroupShuffleSplit

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import callbacks

spotify = pd.read_csv('../input/dl-course-data/spotify.csv')

X = spotify.copy().dropna()
y = X.pop('track_popularity')
artists = X['track_artist']

features_num = ['danceability', 'energy', 'key', 'loudness', 'mode',
                'speechiness', 'acousticness', 'instrumentalness',
                'liveness', 'valence', 'tempo', 'duration_ms']
features_cat = ['playlist_genre']

preprocessor = make_column_transformer(
    (StandardScaler(), features_num),
    (OneHotEncoder(), features_cat),
)

# Suddividi per gruppi, mantenendo tutte le canzoni di uno stesso artista
# nello stesso insieme, per evitare contaminazioni tra training e validazione.
def group_split(X, y, group, train_size=0.75):
    splitter = GroupShuffleSplit(train_size=train_size)
    train, test = next(splitter.split(X, y, groups=group))
    return (X.iloc[train], X.iloc[test], y.iloc[train], y.iloc[test])

X_train, X_valid, y_train, y_valid = group_split(X, y, artists)

X_train = preprocessor.fit_transform(X_train)
X_valid = preprocessor.transform(X_valid)
y_train = y_train / 100 # La popolarità va da 0 a 100: questa operazione la riporta tra 0 e 1.
y_valid = y_valid / 100

input_shape = [X_train.shape[1]]
print("Input shape: {}".format(input_shape))

Partiamo dalla rete più semplice, un modello lineare con capacità limitata. Esegui la cella senza modifiche per addestrarlo.


In [ ]:
model = keras.Sequential([
    layers.Dense(1, input_shape=input_shape),
])
model.compile(
    optimizer='adam',
    loss='mae',
)
history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    batch_size=512,
    epochs=50,
    verbose=0, # Nascondi l'output: i risultati verranno mostrati nei grafici.
)
history_df = pd.DataFrame(history.history)
history_df.loc[0:, ['loss', 'val_loss']].plot()
print("Minimum Validation Loss: {:0.4f}".format(history_df['val_loss'].min()));

Le curve possono scendere rapidamente all'inizio e poi appiattirsi, nascondendo le variazioni finali. Ridisegniamole a partire dall'epoca 10.


In [ ]:
# Inizia il grafico dall'epoca 10.
history_df.loc[10:, ['loss', 'val_loss']].plot()
print("Minimum Validation Loss: {:0.4f}".format(history_df['val_loss'].min()));

# 1. Valutare la baseline
Il modello mostra underfitting, overfitting o un equilibrio ragionevole? Usa entrambe le curve per rispondere.


In [ ]:
# Mostra la soluzione; esegui questa cella per convalidare l'esercizio.
q_1.check()

Aumentiamo la capacità con **tre layer nascosti da 128 unità**. Esegui la cella e osserva le curve.


In [ ]:
model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=input_shape),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)
])
model.compile(
    optimizer='adam',
    loss='mae',
)
history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    batch_size=512,
    epochs=50,
)
history_df = pd.DataFrame(history.history)
history_df.loc[:, ['loss', 'val_loss']].plot()
print("Minimum Validation Loss: {:0.4f}".format(history_df['val_loss'].min()));

# 2. Aumentare la capacità
Come interpreti ora le curve? Il modello è troppo semplice, si sovradatta oppure generalizza ragionevolmente?


In [ ]:
# Mostra la soluzione; esegui questa cella per convalidare l'esercizio.
q_2.check()

# 3. Definire EarlyStopping
Crea una callback con **`patience=5`**, **`min_delta=0.001`** e **`restore_best_weights=True`**: deve aspettare cinque epoche senza miglioramenti sufficienti e recuperare i pesi migliori.


In [ ]:
from tensorflow.keras import callbacks

# SCRIVI QUI IL CODICE: definisci una callback per l'early stopping.
early_stopping = ____

# Controlla la risposta.
q_3.check()

In [ ]:
# Le righe seguenti mostrano un suggerimento o il codice della soluzione.
#q_3.hint()
#q_3.solution()

Addestra il modello e disegna le curve. Osserva come la callback viene passata a **`model.fit`**.


In [ ]:
model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=input_shape),
    layers.Dense(64, activation='relu'),    
    layers.Dense(1)
])
model.compile(
    optimizer='adam',
    loss='mae',
)
history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    batch_size=512,
    epochs=50,
    callbacks=[early_stopping]
)
history_df = pd.DataFrame(history.history)
history_df.loc[:, ['loss', 'val_loss']].plot()
print("Minimum Validation Loss: {:0.4f}".format(history_df['val_loss'].min()));

# 4. Addestrare e interpretare
L'early stopping ha migliorato il risultato rispetto all'esecuzione senza arresto anticipato?


In [ ]:
# Mostra la soluzione; esegui questa cella per convalidare l'esercizio.
q_4.check()

Sperimenta con **`patience`** e **`min_delta`** e descrivi le differenze. Poi studia **[dropout e batch normalization](https://prof-davidaffo.github.io/kaggle-learntools/corsi/deep_learning_intro/tut5.html)**.
